In [34]:
import json
import pandas as pd

rows = []

with open("geschaefte.jsonl", encoding="utf-8") as f:
    for line in f:
        rows.append(json.loads(line))

states= rows[2]["drafts"][0]['states']
state_list = []
for state in states:
    state_list.append(state["id"])
if 222 in state_list:
    success=1
    
    

no


In [41]:
rows[2]

{'id': 20253002,
 'updated': '2025-11-14T03:32:41Z',
 'additionalIndexing': '15;08',
 'affairType': {'abbreviation': 'Po.', 'id': 6, 'name': 'Postulat'},
 'author': {'committee': {'abbreviation': 'APK-NR',
   'id': 4,
   'name': 'Aussenpolitische Kommission NR',
   'abbreviation1': 'APK-N',
   'abbreviation2': 'APK',
   'committeeNumber': 4,
   'council': {'abbreviation': 'NR',
    'id': 1,
    'name': 'Nationalrat',
    'type': 'N'},
   'typeCode': 1},
  'type': 'author'},
 'deposit': {'council': {'abbreviation': 'NR',
   'id': 1,
   'name': 'Nationalrat',
   'type': 'N'},
  'date': '2025-01-13T00:00:00Z',
  'legislativePeriod': 52,
  'session': '5207'},
 'descriptors': [],
 'drafts': [{'consultation': {'resolutions': [{'category': {'id': 3,
       'name': 'Normal'},
      'council': {'abbreviation': 'NR',
       'id': 1,
       'name': 'Nationalrat',
       'type': 'N'},
      'date': '2025-03-20T00:00:00Z',
      'text': 'Annahme',
      'type': 20}]},
   'federalCouncilProposal': {

In [40]:
df_Int = (
    df[df["type"] == "Interpellation"]
  .groupby(["person_code", "author"])
    .size()
)

In [56]:
flat = []

for g in rows:

    if "councillor" not in g.get("author", {}):
        continue

    states = g["drafts"][0]["states"]

    state_list = []
    for state in states:
        state_list.append(state["id"])

    if 209 in state_list:
        success = 1
    else:
        success = 0

    flat.append({
        "person_code": g["author"]["councillor"]["code"],
        "author": g["author"]["councillor"]["name"],
        "type": g["affairType"]["name"],
        "success": success
    })

df = pd.DataFrame(flat)
df["success"].value_counts()

success
0    4174
1     282
Name: count, dtype: int64

In [67]:
df_Int = (
    df[df["type"] == "Interpellation"]
    .groupby(["person_code", "author"])
    .size()
)

df_Post = (
    df[df["type"] == "Postulat"]
    .groupby(["person_code", "author"])
    .size()
)

df_Mot = (
    df[df["type"] == "Motion"]
    .groupby(["person_code", "author"])
    .size()
)

df_tot = (
    df.groupby(["person_code", "author"])
    .size()
)

df_success = (
    df.groupby(["person_code", "author"])["success"]
    .sum()
)

df_stats = pd.DataFrame({
    "total": df_tot,
    "motionen": df_Mot,
    "postulate": df_Post,
    "interpellationen": df_Int,
    "success": df_success
}).fillna(0)

In [73]:
df_stats = pd.DataFrame({
    "total": df_tot,
    "motionen": df_Mot,
    "postulate": df_Post,
    "interpellationen": df_Int,
    "success":df_success,
    "suc_percent": 100/df_tot*df_success
}).fillna(0)

In [74]:
df_stats = df_stats.reset_index()

In [75]:
df_stats.sort_values("success", ascending=False).head(20)

,person_code,author,total,motionen,postulate,interpellationen,success,suc_percent
190,10805,Broulis Pascal,42,12.0,6.0,24.0,11,26.190476
124,3153,Friedli Esther,21,15.0,3.0,3.0,9,42.857143
33,2776,Regazzi Fabio,33,11.0,5.0,17.0,8,24.242424
110,3136,Würth Benedikt,18,11.0,3.0,4.0,7,38.888889
32,2775,Caroni Andrea,10,5.0,3.0,2.0,6,60.000000
160,3198,Z'graggen Heidi,25,11.0,5.0,9.0,6,24.000000
114,3141,Binder-Keller Marianne,22,9.0,7.0,6.0,5,22.727273
81,3092,Salzmann Werner,17,10.0,2.0,5.0,5,29.411765
164,3205,Michel Matthias,12,5.0,2.0,5.0,4,33.333333
7,2632,Sommaruga Carlo,26,12.0,5.0,9.0,4,15.384615


In [76]:
import json

liste = df_stats.to_dict(orient="records")

with open("parlamentarier_stats.json", "w", encoding="utf-8") as f:
    json.dump(liste, f, ensure_ascii=False, indent=2)